In [1]:
import numpy as np

# -----------------------------
# Material / geometry
# -----------------------------
E = 4.32e8          # lb/ft^2
t = 384.0           # thickness (ft)
A = t               # unit width assumption (ft^2)
I = t**3 / 12       # ft^4

# Arch geometry
R = 500.0           # radius (ft)
span = 400.0        # ft
n_nodes = 9
n_elem = n_nodes - 1

# Compute angle range for given span
theta_max = np.arcsin(span / (2 * R))
theta_vals = np.linspace(-theta_max, theta_max, n_nodes)

# Node coordinates (circular arch)
x = R * np.sin(theta_vals)
y = R * (1 - np.cos(theta_vals))

# -----------------------------
# Hydrostatic load
# -----------------------------
p = 24964.0         # psf
w = p * t           # lb/ft (line load)

# -----------------------------
# DOF setup
# -----------------------------
ndof = 3 * n_nodes
K = np.zeros((ndof, ndof))
F = np.zeros(ndof)

# -----------------------------
# Frame element stiffness (local)
# -----------------------------
def frame_element_stiffness(E, A, I, L):
    return np.array([
        [ A*E/L,        0,              0,         -A*E/L,       0,              0],
        [ 0,      12*E*I/L**3,   6*E*I/L**2,       0,     -12*E*I/L**3,   6*E*I/L**2],
        [ 0,      6*E*I/L**2,    4*E*I/L,          0,     -6*E*I/L**2,    2*E*I/L],
        [-A*E/L,       0,              0,          A*E/L,        0,              0],
        [ 0,     -12*E*I/L**3,  -6*E*I/L**2,       0,      12*E*I/L**3,  -6*E*I/L**2],
        [ 0,      6*E*I/L**2,    2*E*I/L,          0,     -6*E*I/L**2,    4*E*I/L]
    ])

# -----------------------------
# Transformation matrix
# -----------------------------
def transformation_matrix(c, s):
    return np.array([
        [ c,  s, 0,  0, 0, 0],
        [-s,  c, 0,  0, 0, 0],
        [ 0,  0, 1,  0, 0, 0],
        [ 0,  0, 0,  c, s, 0],
        [ 0,  0, 0, -s, c, 0],
        [ 0,  0, 0,  0, 0, 1]
    ])

# -----------------------------
# Assembly
# -----------------------------
for e in range(n_elem):

    # Node indices
    n1 = e
    n2 = e + 1

    x1, y1 = x[n1], y[n1]
    x2, y2 = x[n2], y[n2]

    dx = x2 - x1
    dy = y2 - y1
    L = np.sqrt(dx**2 + dy**2)

    c = dx / L
    s = dy / L

    k_local = frame_element_stiffness(E, A, I, L)
    T = transformation_matrix(c, s)
    k_global = T.T @ k_local @ T

    # DOF indices
    dofs = [
        3*n1, 3*n1+1, 3*n1+2,
        3*n2, 3*n2+1, 3*n2+2
    ]

    # Assemble stiffness
    for i in range(6):
        for j in range(6):
            K[dofs[i], dofs[j]] += k_global[i, j]

    # -----------------------------
    # Load: hydrostatic pressure acts normal to arch
    # -----------------------------
    # Normal vector (pointing inward)
    nx = -s
    ny = c

    # Equivalent nodal load in LOCAL coords (transverse load)
    f_local = np.array([
        0,
        w*L/2,
        w*L**2/12,
        0,
        w*L/2,
        -w*L**2/12
    ])

    # Transform to global
    f_global = T.T @ f_local

    # Project to global normal direction
    f_global[0] *= nx
    f_global[1] *= ny
    f_global[3] *= nx
    f_global[4] *= ny

    # Assemble force vector
    for i in range(6):
        F[dofs[i]] += f_global[i]

# -----------------------------
# Boundary conditions (fixed-fixed)
# -----------------------------
fixed_dofs = [0,1,2, ndof-3, ndof-2, ndof-1]
free_dofs = np.setdiff1d(np.arange(ndof), fixed_dofs)

K_ff = K[np.ix_(free_dofs, free_dofs)]
F_ff = F[free_dofs]

# Solve
U = np.zeros(ndof)
U[free_dofs] = np.linalg.solve(K_ff, F_ff)

# Reactions
R = K @ U - F

# -----------------------------
# Crown results
# -----------------------------
mid = n_nodes // 2
ux_crown = U[3*mid]
uy_crown = U[3*mid + 1]

print("\nCrown displacement:")
print("ux =", ux_crown, "ft")
print("uy =", uy_crown, "ft")

# -----------------------------
# Internal forces at crown element
# -----------------------------
e = mid - 1
n1 = e
n2 = e + 1

dofs = [
    3*n1, 3*n1+1, 3*n1+2,
    3*n2, 3*n2+1, 3*n2+2
]

x1, y1 = x[n1], y[n1]
x2, y2 = x[n2], y[n2]
L = np.sqrt((x2-x1)**2 + (y2-y1)**2)

c = (x2-x1)/L
s = (y2-y1)/L

k_local = frame_element_stiffness(E, A, I, L)
T = transformation_matrix(c, s)

u_e_global = U[dofs]
u_e_local = T @ u_e_global

f_local = k_local @ u_e_local

N = f_local[0]
V = f_local[1]
M = f_local[2]

print("\nInternal forces at crown:")
print("Axial N =", N)
print("Shear V =", V)
print("Moment M =", M)

# -----------------------------
# Stress at crown
# -----------------------------
c_dist = t / 2

sigma_axial = N / A
sigma_bending = M * c_dist / I

sigma_max = sigma_axial + sigma_bending
sigma_min = sigma_axial - sigma_bending

print("\nPrincipal stresses at crown:")
print("sigma_max =", sigma_max)
print("sigma_min =", sigma_min)


Crown displacement:
ux = 0.032240314745147595 ft
uy = 0.3812688347130105 ft

Internal forces at crown:
Axial N = -178962115.60777807
Shear V = -258293328.22433472
Moment M = 56134575300.043945

Principal stresses at crown:
sigma_max = 1818074.54024846
sigma_min = -2750168.892372304


In [2]:
import numpy as np

# Internal forces at crown
N = -178962115.60777807  # axial lb
V = -258293328.22433472  # shear lb
M = 56134575300.043945   # lb-ft

# Section properties
A = 384.0          # ft^2
I = 384**3 / 12    # ft^4
c = 384 / 2        # ft

# Convert bending moment to stress
sigma_axial = N / A
sigma_bending = M * c / I

# Shear stress (approx, avg across section)
tau_xy = V / A

# Stress components in 2D plane
sigma_x = sigma_axial
sigma_y = 0  # no stress perpendicular to arch in local 2D model
tau_xy = tau_xy

# Principal stresses
sigma_1 = (sigma_x + sigma_y)/2 + np.sqrt(((sigma_x - sigma_y)/2)**2 + tau_xy**2)
sigma_2 = (sigma_x + sigma_y)/2 - np.sqrt(((sigma_x - sigma_y)/2)**2 + tau_xy**2)

# Principal angle (rad, measured from x-axis to sigma_1)
theta_p = 0.5 * np.arctan2(2*tau_xy, sigma_x - sigma_y)  

# Convert to degrees
theta_deg = np.degrees(theta_p)

print("Principal stresses (lb/ft^2):")
print("sigma_1 =", sigma_1)
print("sigma_2 =", sigma_2)
print("Angle of sigma_1 (deg from x-axis) =", theta_deg)

Principal stresses (lb/ft^2):
sigma_1 = 478835.2810758351
sigma_2 = -944882.4571377571
Angle of sigma_1 (deg from x-axis) = -54.55385963486655
